<div align="center">
  <img src="assets/Day8.png" alt="Databricks 14 Days AI Challenge - Day 08" width="800"/>
</div>

## DAY 8 (16/01/26) – Unity Catalog Governance

### 📚 Learning Objectives
We have built a powerful pipeline, but currently, our data just sits in files. To make this "Enterprise Ready," we need **Governance**. Today we implement **Unity Catalog**:
* **The 3-Level Namespace:** `Catalog` → `Schema` (Database) → `Table`. 
* **Access Control:** Granting permissions to specific teams (e.g., "Analysts can read Gold, but not Bronze").
* **Data Masking:** Using Views to hide sensitive columns (like `user_id`) from unauthorized users.

### 🚀 Strategy: "Governance as Code"
1.  **Structure:** Create a dedicated Catalog and Schema for our eCommerce project.
2.  **Register:** Promote our Delta files (Bronze/Silver/Gold) into proper Managed Tables.
3.  **Secure:** Assign specific privileges using SQL `GRANT` commands.
4.  **Curate:** Create a "Safe View" for the marketing team that aggregates data without exposing individual user IDs.

###Catalog & Schema Setup
**Task**: Create the 3-level hierarchy. **Concept**: This acts as the container for all our assets.

* **Catalog**: The top-level container (e.g., prod_catalog).

* **Schema**: The logical grouping (e.g., ecommerce).

In [0]:
# ---------------------------------------------------------
# STEP 1: SETUP HIERARCHY
# Note: In Community Edition, you might strictly use 'hive_metastore'.
# We will check if we can create a custom catalog, otherwise default to hive_metastore.
# ---------------------------------------------------------

# Define our namespace
catalog_name = "course_catalog"  # Or "hive_metastore" if on Community Edition
schema_name = "ecommerce_governed"

print(f"🏗️ Building Architecture: {catalog_name}.{schema_name}")

try:
    # 1. Create Catalog (If permissions allow)
    spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")
    spark.sql(f"USE CATALOG {catalog_name}")
    
    # 2. Create Schema (Database)
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_name}")
    spark.sql(f"USE SCHEMA {schema_name}")
    
    print(f"✅ Context set to: {catalog_name}.{schema_name}")
    
except Exception as e:
    print(f"⚠️ Permission Warning: Could not create catalog. Defaulting to hive_metastore.\n{e}")
    spark.sql("USE CATALOG hive_metastore")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_name}")
    spark.sql(f"USE SCHEMA {schema_name}")

###Registering Managed Tables
**Task**: Convert file paths to Tables. **Concept**: A Managed Table in Unity Catalog means Databricks manages the underlying files. If we drop the table, the data is deleted too. This is safer for production data than external files.

In [0]:
# Define source paths from our previous days (The Delta Files)
base_path = "/Volumes/workspace/ecommerce/ecommerce_data/delta"
paths = {
    "bronze_events": f"{base_path}/bronze_events",
    "silver_events": f"{base_path}/silver_events",
    "gold_product_perf": f"{base_path}/gold_product_perf"
}

# ---------------------------------------------------------
# STEP 2: REGISTER TABLES (CTAS)
# We use "Create Table As Select" to copy our data into the governed system.
# ---------------------------------------------------------

def register_table(table_name, source_path):
    print(f"📦 Registering Table: {table_name}...")
    
    # Read the existing Delta files
    df = spark.read.format("delta").load(source_path)
    
    # Write as a Managed Table in our new Schema
    # saveAsTable is the Pythonic way to do "CREATE TABLE AS SELECT"
    df.write.format("delta").mode("overwrite").saveAsTable(table_name)
    
    print(f"   ✅ Registered {table_name} successfully.")

# Run registration for all 3 layers
register_table("bronze_events", paths["bronze_events"])
register_table("silver_events", paths["silver_events"])
register_table("gold_product_perf", paths["gold_product_perf"])

###Access Control (GRANT/REVOKE)
* **Task**: Set up permissions. **Concept**: Unity Catalog uses standard SQL for permissions. We will simulate a "Marketing Team" group and give them limited access.

In [0]:
# ---------------------------------------------------------
# STEP 3: PERMISSIONS
# Strategy: "Principle of Least Privilege"
# - Bronze: Restricted (Engineers only)
# - Gold: Accessible to Analysts
# ---------------------------------------------------------

# Note: In a real demo, we would create a group 'analysts'. 
# Here we demonstrate the SQL syntax.

query_permissions = """
    -- 1. Grant Read Access on the Gold Table
    GRANT SELECT ON TABLE gold_product_perf TO `users`; -- 'users' is a default group
    
    -- 2. Revoke Access to Raw PII data (Silver) from general users
    REVOKE SELECT ON TABLE silver_events FROM `users`;
"""

# We execute generic SQL for governance tasks as it's the industry standard
print("🔐 Applying Governance Policies...")
# spark.sql(query_permissions) # Uncomment this line in a real UC-enabled workspace
print("   ✅ Permissions script generated (simulated for demo).")

# Verify the tables exist in the catalog
display(spark.sql("SHOW TABLES"))

###Secure Views (Data Masking)
**Task**: Create views for controlled access. **Concept**: We need to let the Marketing team see purchase trends without seeing exact User IDs. We use a View to aggregate and mask data on the fly.

In [0]:
from pyspark.sql.functions import col, sha2, concat_ws, lit

# ---------------------------------------------------------
# STEP 4: CREATE SECURE VIEW
# Strategy: Create a dynamic view that hashes user_id before displaying it.
# This allows distinct counting but prevents identifying the user.
# ---------------------------------------------------------

# 1. Load Silver Data (for preview, not for view creation)
df_silver = spark.read.table("silver_events")

# 2. Apply Masking Logic (SHA-256 Hashing) for preview only
df_masked = df_silver.select(
    col("event_time"),
    col("event_type"),
    col("product_id"),
    col("price"),
    col("brand"),
    sha2(col("user_id").cast("string"), 256).alias("hashed_user_id")
)

# 3. Create the View directly from the table using SQL
view_name = "marketing_safe_view"
spark.sql(f"""
    CREATE OR REPLACE VIEW {schema_name}.{view_name} AS 
    SELECT 
        event_time,
        event_type,
        product_id,
        price,
        brand,
        sha2(CAST(user_id AS STRING), 256) AS hashed_user_id
    FROM silver_events
""")

print(f"🛡️ Secure View Created: {schema_name}.{view_name}")

# ---------------------------------------------------------
# VISUALIZATION
# Prove that the data is readable but secure
# ---------------------------------------------------------
print("📊 Previewing Secure Data for Marketing Team:")
display(spark.read.table(f"{schema_name}.{view_name}").limit(3))


### 🧠 Key Learnings & Takeaways
* **Hierarchy is Power:** Organizing data into `Catalog.Schema.Table` enables clear ownership boundaries (e.g., keeping "HR" data separate from "Sales" data).
* **Managed vs. External:** We promoted our external files to **Managed Tables**, giving Unity Catalog full lifecycle control (safer for production).
* **Security at the Source:** Instead of creating separate "sanitized CSVs" for analysts, we created a **View**. This ensures the downstream team always sees live data, but with the sensitive columns masked automatically.